In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ===== RUTAS =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
h3_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H3")
h3_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
macro = pd.read_csv(base_dir / "Dataset Maestro.csv")

# ===== TIPOS =====
macro["price"] = pd.to_numeric(macro["price"], errors="coerce")
macro["rating"] = pd.to_numeric(macro["rating"], errors="coerce")
macro["review_count"] = pd.to_numeric(macro["review_count"], errors="coerce")
macro["discount_pct"] = pd.to_numeric(macro["discount_pct"], errors="coerce")
macro["subcategory"] = macro["subcategory"].astype("string").str.strip()

# ===== LIMPIEZA =====
df = macro.dropna(subset=["subcategory", "price"]).copy()

# ===== PERCENTILES POR SUBCATEGORÍA =====
percentiles = (
    df.groupby("subcategory")["price"]
    .quantile([0.50, 0.90])
    .unstack()
    .rename(columns={0.50: "p50", 0.90: "p90"})
    .reset_index()
)

df = df.merge(percentiles, on="subcategory", how="left")

def asignar_gama(row):
    if pd.isna(row["price"]) or pd.isna(row["p50"]) or pd.isna(row["p90"]):
        return pd.NA
    if row["price"] >= row["p90"]:
        return "alta"
    elif row["price"] >= row["p50"]:
        return "media"
    else:
        return "baja"

df["price_tier"] = df.apply(asignar_gama, axis=1)

# ===== TOP 10% POPULARIDAD POR SUBCATEGORÍA =====
q90_reviews = (
    df.groupby("subcategory")["review_count"]
    .quantile(0.90)
    .rename("q90_review_count")
    .reset_index()
)

df = df.merge(q90_reviews, on="subcategory", how="left")
df["top_10_popularity"] = df["review_count"] >= df["q90_review_count"]

# ===== TABLA H3: DESCUENTO POR GAMA Y POPULARIDAD =====
h3_resumen = (
    df.groupby(["subcategory", "price_tier", "top_10_popularity"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_discount_pct=("discount_pct", "mean"),
        median_discount_pct=("discount_pct", "median"),
        avg_review_count=("review_count", "mean"),
        avg_rating=("rating", "mean"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

h3_resumen["top_10_popularity"] = h3_resumen["top_10_popularity"].astype(str)

# ===== TABLA EXTRA: DIFERENCIA TOP VS RESTO =====
h3_compare = (
    h3_resumen.pivot_table(
        index=["subcategory", "price_tier"],
        columns="top_10_popularity",
        values="avg_discount_pct"
    )
    .reset_index()
)

# Renombrar columnas si existen
h3_compare.columns = [
    "subcategory" if c == "subcategory" else
    "price_tier" if c == "price_tier" else
    "avg_discount_no" if c == "False" else
    "avg_discount_yes" if c == "True" else c
    for c in h3_compare.columns
]

if "avg_discount_yes" in h3_compare.columns and "avg_discount_no" in h3_compare.columns:
    h3_compare["diff_discount_top_vs_rest"] = h3_compare["avg_discount_yes"] - h3_compare["avg_discount_no"]

# ===== GUARDAR CSV =====
h3_resumen.to_csv(h3_dir / "h3_resumen.csv", index=False, encoding="utf-8-sig")
h3_compare.to_csv(h3_dir / "h3_compare_discount.csv", index=False, encoding="utf-8-sig")

# ===== FUNCIÓN PNG =====
def save_table_png(df_table, filename, figsize=None):
    if figsize is None:
        figsize = (14, max(4, 0.35 * len(df_table) + 1.5))
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    table = ax.table(cellText=df_table.values, colLabels=df_table.columns, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.25)
    plt.tight_layout()
    fig.savefig(h3_dir / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)

# ===== GUARDAR PNG =====
save_table_png(h3_resumen, "h3_resumen.png")
save_table_png(h3_compare.fillna(""), "h3_compare_discount.png", figsize=(16, 12))

print(f"✅ Archivos H3 guardados en: {h3_dir}")
print(h3_resumen.head())

✅ Archivos H3 guardados en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H3
  subcategory price_tier top_10_popularity  n_products  avg_discount_pct  \
0      Cascos       alta              True           4         50.000000   
1      Cascos       baja              True          20          5.555556   
2      Cascos      media              True          16          0.000000   
3      Mandos       alta              True           5         60.000000   
4      Mandos       baja              True          19          0.000000   

   median_discount_pct  avg_review_count  avg_rating      avg_price  
0                 50.0              39.0         NaN  186240.000000  
1                  0.0              39.0         NaN   26522.000000  
2                  0.0              39.0         NaN   78110.625000  
3                 80.0             780.0         NaN  101590.000000  
4                  0.0             780.0         NaN   23374.210526  
